# nb-03 · Cleanup  *(stage: CLEANUP / TIDY)*

**Purpose.** Route every fetched response through its per-source parser, reshape to
**tidy long**, harmonize units, preserve QA flags, validate against the pandera
schema, and write the deliverable CSV + provenance sidecar.

| | |
|---|---|
| **Inputs** | `data/original/` (from nb-01) · parsers in `src/reservoir/parsers/` |
| **Outputs** | `data/processed/reservoir-storage.csv` · `data/processed/provenance.csv` · `data/audit/extraction_errors.json` (if any) |
| **Preconditions** | nb-01 has run. |

> **Tidy long** = one row per `(source, reservoir_id, datetime, variable)`.
> **Errors are durable, not fatal** — a failed parse lands in `extraction_errors.json`
> and the run continues; `fail_on_empty=True` makes a zero-row result a hard error.

In [1]:
# Make the `reservoir` package importable when running from notebooks/.
# Cleaner: `uv pip install -e ..` once, then this cell is a no-op.
import sys, pathlib
SRC = pathlib.Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('package path:', SRC)

package path: /Users/briankeegan/Documents/GitHub/co-environmental-data/pipelines/reservoir-storage/src


In [2]:
from reservoir import clean, config
import pandas as pd
data, prov = clean.run(fail_on_empty=False)   # set True once live data flows
print('rows:', len(data), '| reservoirs:', data['reservoir_id'].nunique(),
      '| sources:', data['source'].nunique())
data.head(10)

rows: 3 | reservoirs: 1 | sources: 1


,source,vintage,reservoir_id,reservoir_name,datetime,variable,value,unit,qa_flag,concept
0,dwr_cdss,current,GREEN MOUNTAIN,Green Mountain Reservoir,2026-06-15,storage_af,138010.0,acre-ft,A,reservoir.storage_af
1,dwr_cdss,current,GREEN MOUNTAIN,Green Mountain Reservoir,2026-06-16,storage_af,138120.0,acre-ft,A,reservoir.storage_af
2,dwr_cdss,current,GREEN MOUNTAIN,Green Mountain Reservoir,2026-06-17,storage_af,138214.0,acre-ft,A,reservoir.storage_af


### Provenance sidecar (one row per `(source, vintage)`)

In [3]:
prov

,source,vintage,source_url,retrieved_at,sha256,extraction_quality,extraction_notes,parser_module,source_license
0,dwr_cdss,current,https://dwr.state.co.us/Rest/GET/api/v2/teleme...,2026-06-18T19:23:13.692909Z,f265b6cfd25ac4cd1df1f687d32b0aa9472d6f78d99fb9...,clean,,reservoir.parsers.dwr_cdss,"Public / 'as-is' (Colorado DWR; no warranty, i..."


### Cleaning pipeline (what `clean.run` enforced)

profile → structural fixes → **deduplication** (composite-key, keep last) →
missing-value treatment (NA preserved, never zero-filled) → **normalization**
(units from `variable`; UTC datetimes) → **validation** (pandera `CanonicalLong`,
`strict=True`, composite-key uniqueness) → reject port. The schema *is* the
contract; a frame that violates it raises here, not downstream.

In [4]:
errs = config.AUDIT / 'extraction_errors.json'
if errs.exists():
    print('extraction errors logged:'); print(errs.read_text())
else:
    print('no extraction errors ✓')
print('schema validation passed ✓  (clean.run validates before writing)')
print('\nnext → nb-04-publish-csv')

no extraction errors ✓
schema validation passed ✓  (clean.run validates before writing)

next → nb-04-publish-csv
